# Taxi Data Pipeline
This notebook cleans raw taxi data and prepares metrics for the dashboard.

In [15]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Simple string paths (going up one folder to reach the root)
dataset_folder = "../dataset/"
data_folder = "../data/"
models_folder = "../models_saved/"

yellow_files = ["yellow_tripdata_2025-01.parquet", "yellow_tripdata_2025-02.parquet", "yellow_tripdata_2025-03.parquet"]
green_files = ["green_tripdata_2025-01.parquet", "green_tripdata_2025-02.parquet", "green_tripdata_2025-03.parquet"]

In [16]:
final_df = pd.DataFrame()

for file in yellow_files + green_files:
    print(f"Processing: {file}")
    df = pd.read_parquet(dataset_folder + file)
    
    # Handle different column names for Green taxis
    if "lpep_pickup_datetime" in df.columns:
        df = df.rename(columns={"lpep_pickup_datetime": "tpep_pickup_datetime", "lpep_dropoff_datetime": "tpep_dropoff_datetime"})
        df["is_yellow"] = 0
    else:
        df["is_yellow"] = 1
        
    # Simple sampling and cleaning
    df = df.sample(n=min(50000, len(df)), random_state=42)
    df = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance'])
    
    # Time calculations
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['trip_duration'] = ((df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60).round(3)
    
    # Remove outliers
    df = df[(df['trip_duration'] > 1) & (df['trip_duration'] < 120)]
    df = df[(df['trip_distance'] > 0.1) & (df['trip_distance'] < 100)]
    
    final_df = pd.concat([final_df, df], ignore_index=True)

print("Final Combined Shape:", final_df.shape)
final_df.head()

Processing: yellow_tripdata_2025-01.parquet
Processing: yellow_tripdata_2025-02.parquet
Processing: yellow_tripdata_2025-03.parquet
Processing: green_tripdata_2025-01.parquet
Processing: green_tripdata_2025-02.parquet
Processing: green_tripdata_2025-03.parquet
Final Combined Shape: (277612, 24)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_yellow,trip_duration,ehail_fee,trip_type
0,2,2025-01-18 20:53:30,2025-01-18 21:00:47,1.0,0.97,1.0,N,238,166,1.0,...,0.0,1.0,13.32,0.0,0.0,0.00,1,7.283,NaN,NaN
1,1,2025-01-25 11:12:51,2025-01-25 11:17:57,1.0,0.60,1.0,N,50,48,2.0,...,0.0,1.0,10.55,2.5,0.0,0.75,1,5.100,NaN,NaN
2,1,2025-01-21 15:09:31,2025-01-21 15:19:02,1.0,0.80,1.0,N,236,237,1.0,...,0.0,1.0,16.65,2.5,0.0,0.00,1,9.517,NaN,NaN
3,2,2025-01-11 22:25:45,2025-01-11 22:34:22,2.0,1.93,1.0,N,231,68,1.0,...,0.0,1.0,20.37,2.5,0.0,0.75,1,8.617,NaN,NaN
4,2,2025-01-04 23:37:07,2025-01-04 23:45:58,NaN,4.44,NaN,None,137,88,0.0,...,0.0,1.0,35.12,NaN,NaN,0.00,1,8.850,NaN,NaN


In [17]:
# Simple time features
final_df['pickup_hour'] = final_df['tpep_pickup_datetime'].dt.hour
final_df['pickup_day'] = final_df['tpep_pickup_datetime'].dt.day_name()
final_df['is_peak'] = final_df['pickup_hour'].isin([7,8,9,17,18,19]).astype(int)
final_df['is_weekend'] = final_df['pickup_day'].isin(['Saturday', 'Sunday']).astype(int)

# Speed and Traffic logic
final_df['speed'] = final_df['trip_distance'] / (final_df['trip_duration'] / 60)
final_df['traffic_index'] = final_df['trip_duration'] / final_df['trip_distance']
final_df['traffic_level'] = pd.cut(
    final_df['traffic_index'],
    bins=[0, 1, 2, 5, 100],
    labels=['low', 'moderate', 'high', 'extreme']
)

final_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,trip_duration,ehail_fee,trip_type,pickup_hour,pickup_day,is_peak,is_weekend,speed,traffic_index,traffic_level
0,2,2025-01-18 20:53:30,2025-01-18 21:00:47,1.0,0.97,1.0,N,238,166,1.0,...,7.283,NaN,NaN,20,Saturday,0,1,7.991212,7.508247,extreme
1,1,2025-01-25 11:12:51,2025-01-25 11:17:57,1.0,0.60,1.0,N,50,48,2.0,...,5.100,NaN,NaN,11,Saturday,0,1,7.058824,8.500000,extreme
2,1,2025-01-21 15:09:31,2025-01-21 15:19:02,1.0,0.80,1.0,N,236,237,1.0,...,9.517,NaN,NaN,15,Tuesday,0,0,5.043606,11.896250,extreme
3,2,2025-01-11 22:25:45,2025-01-11 22:34:22,2.0,1.93,1.0,N,231,68,1.0,...,8.617,NaN,NaN,22,Saturday,0,1,13.438552,4.464767,high
4,2,2025-01-04 23:37:07,2025-01-04 23:45:58,NaN,4.44,NaN,None,137,88,0.0,...,8.850,NaN,NaN,23,Saturday,0,1,30.101695,1.993243,moderate


In [18]:
# Load zone lookup
zone_df = pd.read_csv(dataset_folder + "taxi_zone_lookup.csv").fillna("Unknown")
zone_map = zone_df.set_index('LocationID')

# Map boroughs and zones
final_df['pickup_borough'] = final_df['PULocationID'].map(zone_map['Borough'])
final_df['pickup_zone'] = final_df['PULocationID'].map(zone_map['Zone'])
final_df['drop_borough'] = final_df['DOLocationID'].map(zone_map['Borough'])
final_df['drop_zone'] = final_df['DOLocationID'].map(zone_map['Zone'])

# Simple zone features
final_df['is_airport'] = final_df['pickup_zone'].str.contains('Airport', case=False).astype(int)
final_df['is_manhattan'] = (final_df['pickup_borough'] == 'Manhattan').astype(int)

final_df[['pickup_zone', 'pickup_borough', 'is_airport']].head()

,pickup_zone,pickup_borough,is_airport
0,Upper West Side North,Manhattan,0
1,Clinton West,Manhattan,0
2,Upper East Side North,Manhattan,0
3,TriBeCa/Civic Center,Manhattan,0
4,Kips Bay,Manhattan,0


In [19]:
# Route logic
final_df['route'] = final_df['pickup_zone'] + " -> " + final_df['drop_zone']

# Corridor Stats
corridor_stats = final_df.groupby('route')['trip_duration'].agg(['mean', 'std']).reset_index()
corridor_stats.columns = ['route', 'corridor_avg_duration', 'corridor_std_duration']
final_df = final_df.merge(corridor_stats, on='route', how='left')
final_df['corridor_volatility'] = final_df['corridor_std_duration'].fillna(0)

# Expected Price logic
zone_avg_price = final_df.groupby('pickup_zone')['total_amount'].transform('mean')
final_df['expected_price'] = (final_df['trip_distance'] * zone_avg_price / final_df['trip_duration'].replace(0,1))

final_df[['route', 'corridor_avg_duration', 'expected_price']].head()

,route,corridor_avg_duration,expected_price
0,Upper West Side North -> Morningside Heights,6.894536,2.723759
1,Clinton West -> Clinton East,6.915261,2.440469
2,Upper East Side North -> Upper East Side South,8.109956,1.710473
3,TriBeCa/Civic Center -> East Chelsea,14.331586,5.606896
4,Kips Bay -> Financial District South,12.945455,10.519262


In [20]:
def explain_trip(row):
    reasons = []
    if row['is_peak'] == 1: reasons.append("Peak hour")
    if row['traffic_index'] > 2: reasons.append("High traffic")
    if row['is_airport'] == 1: reasons.append("Airport trip")
    if row['corridor_volatility'] > 10: reasons.append("Unstable route")
    
    if len(reasons) == 0: return "Normal conditions"
    return ", ".join(reasons)

final_df['trip_explanation'] = final_df.apply(explain_trip, axis=1)
final_df[['route', 'trip_explanation']].head()

,route,trip_explanation
0,Upper West Side North -> Morningside Heights,High traffic
1,Clinton West -> Clinton East,High traffic
2,Upper East Side North -> Upper East Side South,High traffic
3,TriBeCa/Civic Center -> East Chelsea,High traffic
4,Kips Bay -> Financial District South,Normal conditions


In [21]:
print("Training models...")

features = ['trip_distance', 'pickup_hour', 'is_peak', 'is_weekend', 'is_airport', 'is_yellow']
X = final_df[features].fillna(0)
y = final_df['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train simple models
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10)
rf_model.fit(X_train, y_train)

# Save the model
joblib.dump(rf_model, models_folder + "RandomForest.pkl")
print("Model saved to models_saved/RandomForest.pkl")

Training models...
Model saved to models_saved/RandomForest.pkl


In [22]:
import os
os.makedirs(data_folder, exist_ok=True)

# Save the cleaned data for the dashboard
final_df.to_parquet(data_folder + "taxi_clean.parquet", engine='fastparquet', index=False)

# Zone Metrics - grouped with correct column names for the backend
zone_metrics = final_df.groupby(['pickup_zone', 'pickup_borough', 'pickup_hour']).agg({
    'trip_duration': 'mean',
    'total_amount': 'mean',
    'speed': 'mean',
    'VendorID': 'count'
}).reset_index().rename(columns={
    'VendorID': 'trip_count',
    'trip_duration': 'avg_duration',
    'total_amount': 'avg_price',
    'speed': 'avg_speed'
})
zone_metrics.to_parquet(data_folder + "zone_metrics.parquet", engine='fastparquet', index=False)

# Corridor Metrics - grouped with correct column names for the backend
corridor_metrics = final_df.groupby(['route', 'pickup_hour']).agg({
    'trip_duration': 'mean',
    'total_amount': 'mean',
    'speed': 'mean',
    'VendorID': 'count'
}).reset_index().rename(columns={
    'VendorID': 'trip_count',
    'trip_duration': 'avg_duration',
    'total_amount': 'avg_price',
    'speed': 'avg_speed'
})
corridor_metrics.to_parquet(data_folder + "corridor_metrics.parquet", engine='fastparquet', index=False)

# Zone Summary for Nearby Price feature
zone_summary = final_df.groupby(['pickup_zone', 'pickup_borough']).agg({
    'total_amount': 'mean',
    'trip_duration': 'mean',
    'speed': 'mean',
    'VendorID': 'count'
}).reset_index().rename(columns={
    'VendorID': 'trip_count',
    'total_amount': 'avg_price',
    'trip_duration': 'avg_duration',
    'speed': 'avg_speed'
})
zone_summary.to_parquet(data_folder + "zone_summary.parquet", engine='fastparquet', index=False)

print("All files saved successfully!")
print(f"  taxi_clean.parquet: {final_df.shape}")
print(f"  taxi_clean.parquet: {final_df}")
print(f"  zone_metrics.parquet: {zone_metrics.shape}")
print(f"  zone_metrics.parquet: {zone_metrics}")
print(f"  corridor_metrics.parquet: {corridor_metrics.shape}")
print(f"  corridor_metrics.parquet: {corridor_metrics}")
print(f"  zone_summary.parquet: {zone_summary.shape}")
print(f"  zone_summary.parquet: {zone_summary}")

All files saved successfully!
  taxi_clean.parquet: (277612, 43)
  taxi_clean.parquet:         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0              2  2025-01-18 20:53:30   2025-01-18 21:00:47              1.0   
1              1  2025-01-25 11:12:51   2025-01-25 11:17:57              1.0   
2              1  2025-01-21 15:09:31   2025-01-21 15:19:02              1.0   
3              2  2025-01-11 22:25:45   2025-01-11 22:34:22              2.0   
4              2  2025-01-04 23:37:07   2025-01-04 23:45:58              NaN   
...          ...                  ...                   ...              ...   
277607         2  2025-03-04 16:11:46   2025-03-04 16:29:43              1.0   
277608         2  2025-03-26 16:15:55   2025-03-26 16:25:05              1.0   
277609         2  2025-03-02 11:29:31   2025-03-02 11:43:22              1.0   
277610         2  2025-03-08 18:51:15   2025-03-08 19:04:33              2.0   
277611         2  2025-03-29 08:0